# HBOT Treatment Target Knowledge Assembly on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kgrid-objects/FAIR-DO-Workshop/blob/main/HBOT3-KA/auxiliary/aux-notebook/hbot_treatment_target_ka_colab.ipynb)

This notebook clones the repository and runs the real, governed KA implementation from `src/orchestrator.js`, which orchestrates the four constituent Knowledge Objects (Wagner, HBOT Decision, Burden, Margolis) exactly as implemented in this package — no reimplemented logic, and no hardcoded questionnaire answers.

`executeKnowledgeAssembly(request)` is called with no `wagnerAskYesNo`/`burdenAskQuestion` overrides, so the Wagner and Burden KOs fall back to their own default interactive prompters (the same `readline`-based prompters used by their own `wagner_colab.ipynb`/`dfu-hbot_colab.ipynb` notebooks), asking each question directly on stdin/stdout when you run `!node KAInvocation.js`.

In [ ]:
# Setup (run once)
!git clone https://github.com/kgrid-objects/FAIR-DO-Workshop.git

%cd /content/FAIR-DO-Workshop/collection/HBOT-Regimen-Burden-KO
!npm install

%cd /content/FAIR-DO-Workshop/HBOT3-KA
!node -v

In [ ]:
# Create a local KAInvocation.js that imports executeKnowledgeAssembly and runs it interactively.
%%bash
cat > KAInvocation.js <<'EOF'
const { executeKnowledgeAssembly } = require('./src/orchestrator');

// Only subject binding, HBOT case assertions, and the Margolis first-visit
// assessment are fixed sample data here: neither the HBOT Decision KO nor
// the Margolis KO owns an interactive collection capability, so the KA
// never derives these from a questionnaire either (Section 2.5/2.6).
const request = {
  request_id: 'req-colab-demo',
  requested_at: '2026-09-24T12:00:00Z',
  index_time: '2026-09-24T10:00:00Z',
  subject_binding: {
    subject_identifier: { system: 'https://example.org/mrn', value: 'MRN-001' },
    ulcer_identifier: { system: 'https://example.org/ulcer', value: 'ULCER-001' },
    care_episode_identifier: { system: 'https://example.org/episode', value: 'EPISODE-001' },
    source_evidence: {}
  },
  hbot_case_assertions: {
    dfu_confirmed: { value: true, source_evidence: {} },
    acute_surgical_intervention: { value: true, source_evidence: {} },
    not_healed_after_30_days: { value: false, source_evidence: {} }
  },
  margolis_first_visit_assessment: {
    wound_area: { value: 1, ucum_code: 'cm2' },
    wound_duration: { value: 4, ucum_code: 'wk' },
    first_visit_at: '2026-09-10T09:00:00Z',
    first_visit_attested: true
  }
};

(async () => {
  console.log('Starting interactive HBOT Treatment Target KA (Wagner then Burden questionnaires)...');
  const result = await executeKnowledgeAssembly(request);

  console.log('\nHBOT Treatment Target KA result:');
  console.log(JSON.stringify(result, null, 2));
})();
EOF

In [ ]:
!node KAInvocation.js